# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and fields using @id
print("Available record sets in the dataset:")
for recset in metadata.record_sets():
    print(f"- @id: {recset['@id']} | Name: {recset.get('name')} | Description: {recset.get('description')}")

# We'll select the main record set for further work, i.e., the one with most fields/tables
record_sets = list(metadata.record_sets())

if len(record_sets) > 0:
    example_record_set_id = record_sets[0]['@id']
    print(f"\nListing fields for record set @id: {example_record_set_id}\n")
    recset_entity = metadata.record_set_by_id(example_record_set_id)
    for field in recset_entity['field']:
        fobj = metadata.field_by_id(field['@id']) if isinstance(field, dict) else metadata.field_by_id(field)
        print(f"- Field @id: {fobj['@id']} | Name: {fobj.get('name')} | DataType: {fobj.get('dataType')}")
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract data from all record sets
# Use @id references only
record_set_ids = [rs['@id'] for rs in metadata.record_sets()]
dataframes = {}

for rsid in record_set_ids:
    print(f"Loading records for record set: {rsid}")
    try:
        data = list(dataset.records(record_set=rsid))
        if data:
            df = pd.DataFrame(data)
            dataframes[rsid] = df
            print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("No records found.")
    except Exception as e:
        print(f"Could not load records for {rsid}: {e}")

# Show head of the first non-empty DataFrame
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id is not None:
    print(f"\nFirst 5 records for record set {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No data was loaded into any DataFrame.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA
# We'll attempt to find a numeric field and group field automatically for demonstration
import numpy as np
df = dataframes.get(main_record_set_id)
if df is not None and not df.empty:
    
    # Find a numeric field by inspecting dtypes or columns names
    numeric_field = None
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not possible_numeric_fields:
        # Try to cast columns that look numeric
        for col in df.columns:
            try:
                converted = pd.to_numeric(df[col], errors='coerce')
                if converted.notna().sum() > 0:
                    df[col] = converted
                    possible_numeric_fields.append(col)
            except Exception:
                pass
    
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        print("No numeric field found. Please review the dataset fields.")
    
    # Pick a group field: the first string column
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field:
            group_field = col
            break
    if group_field:
        print(f"Using group field: {group_field}")
    
    if numeric_field is not None:
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records.")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by group_field (if available)
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
else:
    print("Main DataFrame is empty or not available. Please check data extraction.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        # If the group field has only a few unique values, plot
        unique_groups = df[group_field].nunique()
        if unique_groups > 1 and unique_groups <= 20:
            plt.figure(figsize=(10,5))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f'{numeric_field} by {group_field}')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.